# Kairos OS · Notebook 3: Continual Multi-Task Fine-Tuning v2

Loads **Kairos-CIDA v1** (strong at Math/Logic) and surgically improves
**coding skills** without overwriting what it already knows.

### Three-pillar strategy

| Pillar | What it does |
|---|---|
| **EWC** (Elastic Weight Consolidation) | Quadratic penalty keeps reasoning weights anchored |
| **Task-balanced weighted sampler** | Prevents large code corpora from dominating the gradient |
| **Curriculum warmup** | Code-first phase for rapid early gain before balanced training |

**New coding datasets:** HumanEval · MBPP · CodeAlpaca · EvoEval · APPS  
**Reasoning anchors kept:** GSM8K · ARC-Challenge · Alpaca  
**Backbone:** Frozen fp16 Qwen1.5-7B-Chat — only CIDA + PlanProjector are trained

**Weights in:** `/content/drive/MyDrive/Kairos_CIDA_7B/best`  
**Weights out:** `/content/drive/MyDrive/Kairos_CIDA_7B/continual_best`

## 0 · Install, Clone & Mount Drive

In [ ]:
!pip install -q torch transformers datasets accelerate torchdiffeq
import subprocess, sys, os
repo = 'CIDA-plugin'
if not os.path.exists(repo):
    subprocess.run(['git','clone','--depth','1',
                   'https://github.com/Kairatzh/CIDA-plugin.git'], check=True)
sys.path.insert(0, os.path.abspath(repo))
from google.colab import drive
drive.mount('/content/drive')
print('Setup complete.')


## 1 · Configuration

In [ ]:
import gc, re, json, time, random, math, warnings
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm.auto import tqdm
from cida_plugin import CIDAPlugin, CIDAPluginConfig

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

@dataclass
class ContinualConfig:
    weights_in:  str   = '/content/drive/MyDrive/Kairos_CIDA_7B/best'
    weights_out: str   = '/content/drive/MyDrive/Kairos_CIDA_7B/continual_best'
    # Dataset caps
    gsm8k_cap:       int   = 7000
    arc_cap:         int   = 3000
    mbpp_cap:        int   = 3500
    humaneval_cap:   int   = 164
    codealpaca_cap:  int   = 4000
    evoeval_cap:     int   = 2000
    apps_cap:        int   = 3000
    alpaca_cap:      int   = 2500
    # Sampling weights  (math+logic=0.35, code=0.50, chat=0.15)
    task_weights:    Dict  = None
    # Tokenizer
    max_prompt_tokens: int = 384
    max_answer_tokens: int = 256
    # Training
    epochs:          int   = 2
    batch_size:      int   = 2
    grad_accum:      int   = 8
    max_lr:          float = 8e-5
    min_lr:          float = 5e-7
    weight_decay:    float = 1e-4
    grad_clip:       float = 1.0
    warmup_steps:    int   = 80
    # EWC
    use_ewc:         bool  = True
    ewc_lambda:      float = 4000.0
    ewc_fisher_samples: int = 300
    # Curriculum
    curriculum_warmup_steps: int = 200
    # Evaluation
    eval_every_steps: int = 150
    eval_n_math:      int = 80
    eval_n_code:      int = 40

    def __post_init__(self):
        if self.task_weights is None:
            self.task_weights = {
                'math': 0.22, 'logic': 0.13,
                'mbpp': 0.12, 'humaneval': 0.10,
                'codealpaca': 0.12, 'evoeval': 0.08,
                'apps': 0.08, 'chat': 0.15,
            }

cfg = ContinualConfig()
print('Config OK')
for k, v in asdict(cfg).items():
    if k != 'task_weights': print(f'  {k:<30} = {v}')
print('  task_weights:', cfg.task_weights)


## 2 · Load Frozen LLM + Pre-Trained CIDA Weights

In [ ]:
def flush():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

WEIGHTS_IN = cfg.weights_in
for fname in ['train_info.json', 'kairos_config.json']:
    p = Path(WEIGHTS_IN) / fname
    if p.exists():
        with open(p) as f: _info = json.load(f)
        break
else:
    _info = {}

MODEL_NAME      = _info.get('llm_model_name', 'Qwen/Qwen1.5-7B-Chat')
NUM_PLAN_TOKENS = _info.get('num_plan_tokens', 8)
print(f'Backbone        : {MODEL_NAME}')
print(f'Plan tokens     : {NUM_PLAN_TOKENS}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True,
                                          padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True,
    torch_dtype=torch.float16, device_map='auto',
    output_hidden_states=True,
)
for p in llm.parameters(): p.requires_grad = False
llm.eval()

LLM_HIDDEN   = llm.config.hidden_size
embed_tokens = llm.get_input_embeddings()
for p in embed_tokens.parameters(): p.requires_grad = False

cida_cfg = CIDAPluginConfig.from_pretrained(WEIGHTS_IN)
cida     = CIDAPlugin(cida_cfg)
sd       = torch.load(str(Path(WEIGHTS_IN) / 'pytorch_model.bin'), map_location='cpu')
cida.load_state_dict(sd)
cida     = cida.to(DEVICE).float()

class CIDAPlanProjector(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan_tokens=8):
        super().__init__()
        self.num_plan_tokens = num_plan_tokens
        self.d_llm = d_llm
        d_flat = num_agents * d_cida
        self.projector = nn.Sequential(
            nn.Linear(d_flat, d_flat * 2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(d_flat * 2, num_plan_tokens * d_llm),
            nn.LayerNorm(num_plan_tokens * d_llm),
        )
    def forward(self, s):
        B = s.size(0)
        return self.projector(s.reshape(B, -1).float()).view(B, self.num_plan_tokens, self.d_llm)

plan_projector = CIDAPlanProjector(
    d_cida=cida_cfg.d_hidden, d_llm=LLM_HIDDEN,
    num_agents=cida_cfg.num_agents, num_plan_tokens=NUM_PLAN_TOKENS,
).to(DEVICE)
plan_projector.load_state_dict(
    torch.load(str(Path(WEIGHTS_IN) / 'plan_projector.pt'), map_location=DEVICE)
)

total_trainable = (sum(p.numel() for p in cida.parameters()) +
                   sum(p.numel() for p in plan_projector.parameters()))
print(f'Frozen LLM      : {sum(p.numel() for p in llm.parameters())/1e9:.2f}B params')
print(f'Trainable CIDA  : {total_trainable/1e6:.2f}M params')
print('Pre-trained weights loaded. Only CIDA + PlanProjector receive gradients.')


## 3 · Dataset Loaders

### Coding datasets (new skills)
| Dataset | Focus |
|---|---|
| **HumanEval** (164) | Function synthesis from docstrings — canonical benchmark |
| **MBPP** (3 500) | Python tasks with test cases |
| **CodeAlpaca** (4 000) | Instruction-following + code |
| **EvoEval** (2 000) | Evolved HumanEval variants — harder, more diverse |
| **APPS** (3 000) | Competitive coding, introductory difficulty |

### Reasoning anchors (full retention)
| Dataset | Role |
|---|---|
| **GSM8K** (7 000) | Math reasoning — anchor for EWC Fisher |
| **ARC-Challenge** (3 000) | Logic — anchor |
| **Alpaca** (2 500) | General instruction following |

In [ ]:
_REPL = '\ufffd'
_CTRL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def is_clean(t):
    return isinstance(t,str) and bool(t.strip()) and _REPL not in t and not _CTRL.search(t)

def is_valid_python(code):
    try: compile(code,'<s>','exec'); return True
    except SyntaxError: return False

def make_prompt(q, task):
    SYS = {
        'math':  'You are a precise mathematical reasoner. Solve step by step, end with #### <answer>.',
        'logic': 'You are a logical reasoning expert. Analyse carefully and choose the best answer.',
        'code':  'You are an expert Python programmer. Write clean, correct code inside a ```python block.',
        'chat':  'You are a helpful assistant.',
    }
    s = SYS.get(task, SYS['chat'])
    return (f'<|im_start|>system\n{s}\n<|im_end|>\n'
            f'<|im_start|>user\n{q.strip()}\n<|im_end|>\n'
            f'<|im_start|>assistant\n')

def has_python(text):
    m = re.search(r'```python\s*(.*?)```', text, re.DOTALL)
    raw = m.group(1).strip() if m else text.strip()
    return len(raw) > 20 and is_valid_python(raw)


def load_gsm8k(cap):
    raw = load_dataset('openai/gsm8k','main',split='train')
    out, skip = [], 0
    for ex in tqdm(raw, desc='GSM8K'):
        if len(out) >= cap: break
        q, a = ex['question'].strip(), ex['answer'].strip()
        if not (is_clean(q) and is_clean(a) and '####' in a): skip+=1; continue
        out.append({'prompt': make_prompt(q,'math'), 'answer': a, 'task': 'math'})
    print(f'  GSM8K      -> {len(out):>5} kept | {skip} skipped'); return out

def load_arc(cap):
    raw = load_dataset('allenai/ai2_arc','ARC-Challenge',split='train')
    out, skip = [], 0
    for ex in tqdm(raw, desc='ARC'):
        if len(out) >= cap: break
        q  = ex['question'].strip()
        ch = ex['choices']
        ak = ex['answerKey'].strip().upper()
        if not (is_clean(q) and ak): skip+=1; continue
        ct = '\n'.join(f'{l}. {t}' for l,t in zip(ch['label'],ch['text']))
        try:
            idx = ch['label'].index(ak)
            ans = f'{ak}. {ch["text"][idx]}'
        except: skip+=1; continue
        out.append({'prompt': make_prompt(f'{q}\n{ct}','logic'), 'answer': ans, 'task': 'logic'})
    print(f'  ARC        -> {len(out):>5} kept | {skip} skipped'); return out

def load_mbpp(cap):
    try:
        raw = load_dataset('google-research-datasets/mbpp','sanitized',split='train',trust_remote_code=True)
    except:
        raw = load_dataset('mbpp',split='train',trust_remote_code=True)
    out, skip = [], 0
    for ex in tqdm(raw, desc='MBPP'):
        if len(out) >= cap: break
        q    = (ex.get('text') or ex.get('prompt') or '').strip()
        code = (ex.get('code') or '').strip()
        if not (is_clean(q) and is_clean(code) and is_valid_python(code)): skip+=1; continue
        tests = ex.get('test_list') or []
        tb    = ('\n\n# Tests\n' + '\n'.join(tests)) if tests else ''
        ans   = f'```python\n{code}{tb}\n```'
        out.append({'prompt': make_prompt(q,'code'), 'answer': ans, 'task': 'mbpp'})
    print(f'  MBPP       -> {len(out):>5} kept | {skip} skipped'); return out

def load_humaneval(cap):
    for ds_id in ['openai_humaneval','openai/openai_humaneval']:
        try:
            raw = load_dataset(ds_id, split='test', trust_remote_code=True); break
        except: pass
    else:
        print('  HumanEval  -> unavailable; skipping.'); return []
    out, skip = [], 0
    for ex in tqdm(raw, desc='HumanEval'):
        if len(out) >= cap: break
        prompt_txt = ex.get('prompt','').strip()
        canon      = ex.get('canonical_solution','').strip()
        test_code  = ex.get('test','').strip()
        if not (is_clean(prompt_txt) and is_clean(canon)): skip+=1; continue
        full = prompt_txt + canon
        if not is_valid_python(full): skip+=1; continue
        q   = f'Complete the following Python function:\n\n{prompt_txt}'
        ans = f'```python\n{full}\n\n{test_code}\n```'
        out.append({'prompt': make_prompt(q,'code'), 'answer': ans, 'task': 'humaneval'})
    print(f'  HumanEval  -> {len(out):>5} kept | {skip} skipped'); return out

def load_codealpaca(cap):
    try:
        raw = load_dataset('sahil2801/CodeAlpaca-20k', split='train')
    except:
        print('  CodeAlpaca -> unavailable; skipping.'); return []
    out, skip = [], 0
    for ex in tqdm(raw, desc='CodeAlpaca'):
        if len(out) >= cap: break
        instr = (ex.get('instruction') or '').strip()
        outp  = (ex.get('output') or '').strip()
        if not (is_clean(instr) and is_clean(outp)): skip+=1; continue
        if '```python' not in outp and 'def ' not in outp: skip+=1; continue
        if not has_python(outp): skip+=1; continue
        out.append({'prompt': make_prompt(instr,'code'), 'answer': outp, 'task': 'codealpaca'})
    print(f'  CodeAlpaca -> {len(out):>5} kept | {skip} skipped'); return out

def load_evoeval(cap):
    SUBSETS = ['difficult','creative','subtle','combine','tool_use']
    out, skip = [], 0
    per_sub   = max(1, cap // len(SUBSETS))
    for sub in SUBSETS:
        if len(out) >= cap: break
        try:
            raw = load_dataset('evoeval/EvoEval', sub, split='test', trust_remote_code=True)
        except: continue
        sub_cnt = 0
        for ex in tqdm(raw, desc=f'EvoEval/{sub}'):
            if len(out) >= cap or sub_cnt >= per_sub: break
            pt  = (ex.get('prompt') or '').strip()
            can = (ex.get('canonical_solution') or '').strip()
            if not (is_clean(pt) and is_clean(can)): skip+=1; continue
            full = pt + can
            if not is_valid_python(full): skip+=1; continue
            ans = f'```python\n{full}\n```'
            out.append({'prompt': make_prompt(f'Complete this Python function:\n\n{pt}','code'),
                        'answer': ans, 'task': 'evoeval'})
            sub_cnt += 1
    if out: print(f'  EvoEval    -> {len(out):>5} kept | {skip} skipped')
    else:   print('  EvoEval    -> unavailable; skipping.')
    return out

def load_apps(cap):
    for ds_id, split_name in [('codeparrot/apps','train'),('APPS','train')]:
        try:
            try:
                raw = load_dataset(ds_id,'introductory',split=split_name,trust_remote_code=True)
            except:
                raw = load_dataset(ds_id,split=split_name,trust_remote_code=True)
            break
        except: pass
    else:
        print('  APPS       -> unavailable; skipping.'); return []
    out, skip = [], 0
    for ex in tqdm(raw, desc='APPS'):
        if len(out) >= cap: break
        problem   = (ex.get('question') or ex.get('problem') or '').strip()
        solutions = ex.get('solutions') or '[]'
        if isinstance(solutions, str):
            try: solutions = json.loads(solutions)
            except: solutions = []
        if not isinstance(solutions, list) or not solutions: skip+=1; continue
        code = next((s for s in sorted(solutions, key=len) if is_valid_python(s) and len(s)>30), None)
        if code is None or not is_clean(problem): skip+=1; continue
        ans = f'```python\n{code.strip()}\n```'
        out.append({'prompt': make_prompt(problem[:1200],'code'), 'answer': ans, 'task': 'apps'})
    print(f'  APPS       -> {len(out):>5} kept | {skip} skipped'); return out

def load_alpaca(cap):
    raw = load_dataset('tatsu-lab/alpaca', split='train')
    out, skip = [], 0
    for ex in tqdm(raw, desc='Alpaca'):
        if len(out) >= cap: break
        instr = (ex.get('instruction') or '').strip()
        inp   = (ex.get('input') or '').strip()
        outp  = (ex.get('output') or '').strip()
        if not (is_clean(instr) and is_clean(outp)): skip+=1; continue
        q = f'{instr}\n{inp}' if inp else instr
        out.append({'prompt': make_prompt(q,'chat'), 'answer': outp, 'task': 'chat'})
    print(f'  Alpaca     -> {len(out):>5} kept | {skip} skipped'); return out

print('All dataset loaders defined.')


## 4 · Execute All Loaders & Build Balanced Dataset

In [ ]:
import random as _rand
_rand.seed(42)

print('='*60)
print('Loading datasets...')
print('='*60)

math_recs  = load_gsm8k(cfg.gsm8k_cap)
logic_recs = load_arc(cfg.arc_cap)
mbpp_recs  = load_mbpp(cfg.mbpp_cap)
he_recs    = load_humaneval(cfg.humaneval_cap)
ca_recs    = load_codealpaca(cfg.codealpaca_cap)
evo_recs   = load_evoeval(cfg.evoeval_cap)
apps_recs  = load_apps(cfg.apps_cap)
chat_recs  = load_alpaca(cfg.alpaca_cap)

all_records = (math_recs + logic_recs + mbpp_recs + he_recs +
               ca_recs + evo_recs + apps_recs + chat_recs)
_rand.shuffle(all_records)

task_counts: Dict[str,int] = {}
for r in all_records:
    task_counts[r['task']] = task_counts.get(r['task'],0) + 1

print(f'\nTotal records: {len(all_records):,}')
for t, c in sorted(task_counts.items()):
    pct = 100*c/len(all_records)
    bar = chr(9608)*int(pct/2)
    print(f'  {t:<12} {c:>6,}  ({pct:5.1f}%)  {bar}')

split_idx     = int(0.92*len(all_records))
train_records = all_records[:split_idx]
val_records   = all_records[split_idx:]
print(f'\nTrain: {len(train_records):,}  |  Val: {len(val_records):,}')


## 5 · Dataset Class + Task-Balanced Weighted Sampler

The `WeightedRandomSampler` draws samples proportional to `cfg.task_weights`,
regardless of raw dataset size. This prevents MBPP/APPS (large) from
drowning out HumanEval (164 samples) or logic anchors.

In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, records: List[dict]):
        self.records = records
    def __len__(self): return len(self.records)
    def __getitem__(self, idx: int) -> dict:
        rec = self.records[idx]
        tokenizer.padding_side = 'left'
        p_enc = tokenizer(rec['prompt'],
                          max_length=cfg.max_prompt_tokens, truncation=True,
                          padding='max_length', return_tensors='pt')
        tokenizer.padding_side = 'right'
        a_enc = tokenizer(rec['answer'],
                          max_length=cfg.max_answer_tokens, truncation=True,
                          padding='max_length', return_tensors='pt')
        tokenizer.padding_side = 'left'
        return {'prompt_ids':  p_enc['input_ids'].squeeze(0),
                'prompt_mask': p_enc['attention_mask'].squeeze(0),
                'answer_ids':  a_enc['input_ids'].squeeze(0),
                'answer_mask': a_enc['attention_mask'].squeeze(0),
                'task':        rec['task']}


def make_weighted_sampler(records, task_weights):
    """Per-sample weight = task_weight / task_count for uniform within-task probability."""
    counts = {}
    for r in records: counts[r['task']] = counts.get(r['task'],0) + 1
    fallback = min(task_weights.values()) if task_weights else 0.1
    weights  = [task_weights.get(r['task'],fallback) / counts.get(r['task'],1)
                for r in records]
    return WeightedRandomSampler(weights=weights, num_samples=len(records),
                                 replacement=True)


train_ds = MultiTaskDataset(train_records)
val_ds   = MultiTaskDataset(val_records)
sampler  = make_weighted_sampler(train_records, cfg.task_weights)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)

# Verify sampler task distribution on 200 draws
_sample_tasks: Dict[str,int] = {}
for _idx in list(sampler)[:200]:
    _t = train_records[_idx]['task']
    _sample_tasks[_t] = _sample_tasks.get(_t,0) + 1
print('Sampler task distribution (first 200 draws):')
for _t, _c in sorted(_sample_tasks.items()):
    print(f'  {_t:<12} {_c:>4}  ({100*_c/200:.1f}%)')
print(f'\nTrain batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')


## 6 · Elastic Weight Consolidation (EWC)

**Why EWC here specifically:**
When we add coding data the gradient signal for math/logic disappears.
EWC adds a penalty `λ · Σ F_i (θ_i - θ*_i)²` that resists changes to parameters
that were critical for the old tasks.

- `θ*` = CIDA weights before continual training (the anchor)
- `F_i` = Fisher information for parameter i, estimated on math+logic data
- `λ = 4000` means any parameter drift costs `4000 × F_i × drift²`

Parameters that are unimportant for reasoning (low Fisher) can freely adapt to coding.

In [ ]:
class EWC:
    """
    Diagonal-Fisher EWC regulariser.
    Usage:
        ewc = EWC(cida, plan_projector, anchor_loader, n_samples, forward_fn)
        penalty = ewc.penalty()   # scalar, multiply by cfg.ewc_lambda
    """
    def __init__(self, cida_m, proj_m, anchor_loader, n_samples, forward_fn):
        self.params  = [p for m in (cida_m, proj_m) for p in m.parameters()
                         if p.requires_grad]
        self._anchor = [p.detach().clone() for p in self.params]
        self._fisher = [torch.zeros_like(p) for p in self.params]
        self._estimate_fisher(anchor_loader, n_samples, forward_fn)

    def _estimate_fisher(self, loader, n_samples, forward_fn):
        print(f'  Estimating Fisher on {n_samples} anchor samples...')
        collected = 0
        for batch in loader:
            if collected >= n_samples: break
            pid  = batch['prompt_ids'].to(DEVICE)
            pmsk = batch['prompt_mask'].to(DEVICE)
            aid  = batch['answer_ids'].to(DEVICE)
            amsk = batch['answer_mask'].to(DEVICE)
            for p in self.params: p.grad = None
            with torch.enable_grad():
                loss, _ = forward_fn(pid, pmsk, aid, amsk)
                loss.backward()
            for i, p in enumerate(self.params):
                if p.grad is not None:
                    self._fisher[i] += p.grad.detach() ** 2
            for p in self.params: p.grad = None
            collected += pid.size(0)
        n = max(collected, 1)
        for i in range(len(self._fisher)): self._fisher[i] /= n
        total_f = sum(f.sum().item() for f in self._fisher)
        print(f'  Fisher estimated. Total mass: {total_f:.4f}')

    def penalty(self) -> torch.Tensor:
        """Returns  Sigma_i  F_i * (theta_i - theta*_i)^2"""
        pen = torch.tensor(0.0, device=DEVICE)
        for p, anc, fsh in zip(self.params, self._anchor, self._fisher):
            pen = pen + (fsh * (p - anc.to(p.device)) ** 2).sum()
        return pen


ewc = None
if cfg.use_ewc:
    print('Building EWC from reasoning anchors...')
    anchor_recs = [r for r in train_records if r['task'] in ('math','logic')]
    _rand.shuffle(anchor_recs)
    anchor_ds   = MultiTaskDataset(anchor_recs[:cfg.ewc_fisher_samples * 4])
    anch_loader = DataLoader(anchor_ds, batch_size=cfg.batch_size,
                             shuffle=True, num_workers=0)
    cida.train(); plan_projector.train()

    # Import forward_fn — defined in next cell, define a stub here
    # (EWC will call it with enable_grad context)
    def _stub_fwd(pid, pmsk, aid, amsk):
        return cida_forward_with_loss(pid, pmsk, aid, amsk)

    print('  (forward function will be defined in next cell; EWC called after)')
    _EWC_DEFERRED = True
else:
    _EWC_DEFERRED = False
    print('EWC disabled.')


## 7 · Forward Pass + Deferred EWC Initialization

The forward pass uses the correct causal shift:
```
logit[Lq+P-1]  ->  predicts  answer_ids[:,0]
answer_logits = logits[:, answer_start-1 : answer_start+La-1, :]
```

In [ ]:
@torch.no_grad()
def get_prompt_context(prompt_ids, prompt_mask):
    out    = llm(input_ids=prompt_ids, attention_mask=prompt_mask,
                 output_hidden_states=True, use_cache=False)
    last_h = out.hidden_states[-1]
    idx    = prompt_mask.sum(1) - 1
    bidx   = torch.arange(prompt_ids.size(0), device=prompt_ids.device)
    pooled = last_h[bidx, idx].float()
    e_dev  = embed_tokens.weight.device
    p_emb  = embed_tokens(prompt_ids.to(e_dev)).to(DEVICE).detach().float()
    return pooled, p_emb


def cida_forward_with_loss(prompt_ids, prompt_mask, answer_ids, answer_mask):
    B, Lq     = prompt_ids.shape
    La        = answer_ids.shape[1]
    P         = NUM_PLAN_TOKENS
    llm_dtype = next(llm.parameters()).dtype

    pooled, p_emb = get_prompt_context(prompt_ids, prompt_mask)
    cida_out      = cida(pooled=pooled)
    plan_emb      = plan_projector(cida_out['state_final'].s)

    e_dev = embed_tokens.weight.device
    with torch.no_grad():
        a_emb = embed_tokens(answer_ids.to(e_dev)).to(DEVICE).float()

    train_emb = torch.cat([p_emb, plan_emb, a_emb], dim=1)
    plan_msk  = torch.ones(B, P, dtype=prompt_mask.dtype, device=DEVICE)
    comb_msk  = torch.cat([prompt_mask, plan_msk, answer_mask], dim=1)

    with torch.cuda.amp.autocast():
        lm_out = llm(inputs_embeds=train_emb.to(llm_dtype),
                     attention_mask=comb_msk, use_cache=False)
    logits = lm_out.logits.float()

    # Correct causal shift: logit[Lq+P-1] predicts answer_ids[:,0]
    answer_start = Lq + P
    ans_logits   = logits[:, answer_start-1 : answer_start+La-1, :]
    loss_mask    = answer_mask.bool()

    B_flat   = B * La
    logits_f = ans_logits.reshape(B_flat, -1)
    tgt_f    = answer_ids.reshape(B_flat).masked_fill(~loss_mask.reshape(B_flat), 0)
    msk_f    = loss_mask.reshape(B_flat)

    loss_tok = F.cross_entropy(logits_f, tgt_f, reduction='none')
    loss     = (loss_tok * msk_f.float()).sum() / msk_f.float().sum().clamp(min=1)
    return loss, cida_out


# Deferred EWC initialization (now that forward_fn is defined)
if _EWC_DEFERRED:
    ewc = EWC(cida_m=cida, proj_m=plan_projector,
              anchor_loader=anch_loader,
              n_samples=cfg.ewc_fisher_samples,
              forward_fn=cida_forward_with_loss)
    print(f'EWC ready. lambda={cfg.ewc_lambda}')

print('Forward pass defined.')


## 8 · Optimizer & Scheduler

In [ ]:
trainable_params = list(cida.parameters()) + list(plan_projector.parameters())
n_trainable      = sum(p.numel() for p in trainable_params)

optimizer = torch.optim.AdamW(trainable_params,
                              lr=cfg.max_lr, weight_decay=cfg.weight_decay,
                              betas=(0.9, 0.98))

total_optim_steps = (len(train_loader) // cfg.grad_accum) * cfg.epochs

def lr_lambda(step):
    if step < cfg.warmup_steps:
        return (step+1) / max(cfg.warmup_steps, 1)
    prog    = (step - cfg.warmup_steps) / max(total_optim_steps - cfg.warmup_steps, 1)
    cosine  = 0.5 * (1.0 + math.cos(math.pi * prog))
    min_f   = cfg.min_lr / cfg.max_lr
    return min_f + (1.0 - min_f) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler(enabled=True)

print(f'Trainable params   : {n_trainable/1e6:.2f}M')
print(f'Optimizer steps    : {total_optim_steps}')
print(f'LR  {cfg.min_lr} -> {cfg.max_lr} -> {cfg.min_lr}  (warmup {cfg.warmup_steps} steps)')
print(f'Effective batch    : {cfg.batch_size * cfg.grad_accum}')


## 9 · Evaluation Helper

In [ ]:
def parse_ans(text):
    m = re.search(r'####\s*([\-0-9,\.]+)', text)
    if m:
        try: return str(int(float(m.group(1).replace(',',''))))
        except: return m.group(1)
    m = re.match(r'^([A-J])[.):\s]', text.strip())
    if m: return m.group(1)
    nums = re.findall(r'\b-?\d+(?:\.\d+)?\b', text.replace(',',''))
    if nums:
        try: return str(int(float(nums[-1])))
        except: return nums[-1]
    return text.strip()[:10]


@torch.no_grad()
def eval_split(records, n=100, max_new=96):
    cida.eval(); plan_projector.eval()
    llm_dtype = next(llm.parameters()).dtype
    task_c: Dict[str,int] = {}
    task_t: Dict[str,int] = {}
    samples = _rand.sample(records, min(n, len(records)))

    def gold_key(rec):
        if rec['task'] == 'math':
            m = re.search(r'####\s*([\-0-9,\.]+)', rec['answer'])
            if m:
                try: return str(int(float(m.group(1).replace(',',''))))
                except: return m.group(1)
        elif rec['task'] == 'logic':
            return rec['answer'].strip().split('.')[0].strip()
        return '__code__'

    for rec in tqdm(samples, desc='Eval', leave=False):
        t    = rec['task']
        gold = gold_key(rec)
        task_t[t] = task_t.get(t,0) + 1

        tokenizer.padding_side = 'left'
        enc = tokenizer(rec['prompt'], return_tensors='pt',
                        max_length=cfg.max_prompt_tokens, truncation=True).to(DEVICE)
        pid, pmsk = enc['input_ids'], enc['attention_mask']

        pooled   = get_prompt_context(pid, pmsk)[0]
        cout     = cida(pooled=pooled)
        plan_emb = plan_projector(cout['state_final'].s)
        e_dev    = embed_tokens.weight.device
        p_emb    = embed_tokens(pid.to(e_dev)).to(DEVICE).float()
        cat_emb  = torch.cat([p_emb, plan_emb], dim=1).to(llm_dtype)
        P        = NUM_PLAN_TOKENS
        ext_msk  = torch.cat([pmsk, torch.ones(1,P,dtype=pmsk.dtype,device=DEVICE)], dim=1)

        gen_ids = llm.generate(inputs_embeds=cat_emb, attention_mask=ext_msk,
                               max_new_tokens=max_new, do_sample=False,
                               pad_token_id=tokenizer.pad_token_id,
                               eos_token_id=tokenizer.eos_token_id)
        gen_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

        if gold == '__code__':
            correct = bool(re.search(r'def |```python', gen_text))
        else:
            pred    = parse_ans(gen_text)
            correct = (pred == gold) or gen_text.strip().startswith(gold)

        if correct: task_c[t] = task_c.get(t,0) + 1

    return {t: task_c.get(t,0)/task_t[t] for t in task_t}

print('Evaluation function defined.')


## 10 · Continual Training Loop

**Training phases:**

1. **Curriculum warmup** (first 200 steps) — code data only, fast early adaptation
2. **Balanced training** — weighted sampler + EWC penalty guards reasoning
3. **Eval every 150 steps** — checks both math acc (must stay up) and code acc (must rise)

In [ ]:
save_dir = Path(cfg.weights_out)
save_dir.mkdir(parents=True, exist_ok=True)

def save_checkpoint(tag=''):
    dst = save_dir/tag if tag else save_dir
    dst.mkdir(parents=True, exist_ok=True)
    cida.save_pretrained(str(dst))
    torch.save(plan_projector.state_dict(), str(dst/'plan_projector.pt'))
    meta = {'llm_model_name': MODEL_NAME, 'num_plan_tokens': NUM_PLAN_TOKENS,
            'cida_d_hidden': cida_cfg.d_hidden, 'llm_hidden': LLM_HIDDEN,
            'stage': 'continual_v2', 'ewc_lambda': cfg.ewc_lambda if cfg.use_ewc else 0.0}
    with open(str(dst/'kairos_config.json'),'w') as f: json.dump(meta,f,indent=2)
    print(f'  Saved -> {dst}')

history = {'step':[], 'task_loss':[], 'ewc_loss':[], 'total_loss':[],
           'lr':[], 'disagreement':[], 'val_step':[], 'val_results':[]}
global_step = 0
best_score  = 0.0

# Curriculum: code-only loader
code_recs = [r for r in train_records if r['task'] not in ('math','logic','chat')]
_rand.shuffle(code_recs)
cur_ds     = MultiTaskDataset(code_recs)
cur_loader = DataLoader(cur_ds, batch_size=cfg.batch_size,
                        shuffle=True, num_workers=2, pin_memory=True)
cur_iter   = iter(cur_loader)

CODE_TASKS = {'mbpp','humaneval','codealpaca','evoeval','apps'}

print(f'Continual fine-tuning: {cfg.epochs} epochs, {total_optim_steps} optim steps')
print(f'  Curriculum warmup : {cfg.curriculum_warmup_steps} steps (code only)')
print(f'  EWC enabled       : {cfg.use_ewc}  (lambda={cfg.ewc_lambda})')
print()

for epoch in range(cfg.epochs):
    cida.train(); plan_projector.train()
    epoch_loss, n_upd = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    t_ep = time.time()

    for batch_idx, batch in enumerate(train_loader):

        # Curriculum warmup override
        if global_step < cfg.curriculum_warmup_steps:
            try:
                batch = next(cur_iter)
            except StopIteration:
                cur_iter = iter(cur_loader)
                batch    = next(cur_iter)

        pid  = batch['prompt_ids'].to(DEVICE)
        pmsk = batch['prompt_mask'].to(DEVICE)
        aid  = batch['answer_ids'].to(DEVICE)
        amsk = batch['answer_mask'].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=True):
            task_loss, cout = cida_forward_with_loss(pid, pmsk, aid, amsk)

        ewc_loss = torch.tensor(0.0, device=DEVICE)
        if cfg.use_ewc and ewc is not None and global_step >= cfg.curriculum_warmup_steps:
            ewc_loss = cfg.ewc_lambda * ewc.penalty()

        total = (task_loss + ewc_loss) / cfg.grad_accum
        scaler.scale(total).backward()
        epoch_loss += total.item() * cfg.grad_accum

        if (batch_idx + 1) % cfg.grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.grad_clip)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1; n_upd += 1

            disag = cout['disagreement'].mean().item() if cout else 0.0
            lr    = scheduler.get_last_lr()[0]
            history['step'].append(global_step)
            history['task_loss'].append(task_loss.item())
            history['ewc_loss'].append(ewc_loss.item())
            history['total_loss'].append(task_loss.item() + ewc_loss.item())
            history['lr'].append(lr)
            history['disagreement'].append(disag)

            if global_step % 20 == 0:
                wtag = ' [WARMUP]' if global_step < cfg.curriculum_warmup_steps else ''
                print(f'  E{epoch+1} step {global_step:>4}/{total_optim_steps} |'
                      f' task={task_loss.item():.4f} ewc={ewc_loss.item():.3f}'
                      f' disag={disag:.3f} lr={lr:.2e}{wtag}')

            if global_step % cfg.eval_every_steps == 0:
                print(f'\n  -- Eval at step {global_step} --')
                cida.eval(); plan_projector.eval()

                m_smp = [r for r in val_records if r['task'] == 'math']
                c_smp = [r for r in val_records if r['task'] in CODE_TASKS]
                m_res = eval_split(m_smp, cfg.eval_n_math, max_new=80)
                c_res = eval_split(c_smp, cfg.eval_n_code, max_new=128)
                all_r = {**m_res, **c_res}
                history['val_step'].append(global_step)
                history['val_results'].append(all_r)

                print('  Results:')
                for t, acc in sorted(all_r.items()):
                    print(f'    {t:<12} {acc*100:6.1f}%')

                m_acc = all_r.get('math', 0.0)
                c_vals = [v for k, v in all_r.items() if k in CODE_TASKS]
                c_acc  = sum(c_vals)/len(c_vals) if c_vals else 0.0
                combo  = 0.5 * m_acc + 0.5 * c_acc
                print(f'  Math={m_acc*100:.1f}%  CodeAvg={c_acc*100:.1f}%  Combined={combo*100:.1f}%')

                if combo >= best_score:
                    best_score = combo
                    save_checkpoint('best')
                    print(f'  New best: {best_score*100:.2f}%')

                cida.train(); plan_projector.train()
                print()

    elapsed = time.time() - t_ep
    print(f'\nEpoch {epoch+1} done {elapsed:.0f}s  '
          f'avg_loss={epoch_loss/max(n_upd*cfg.grad_accum,1):.4f}\n')

save_checkpoint('final')
print(f'Continual fine-tuning complete! Best score: {best_score*100:.2f}%')
print(f'Weights saved to: {cfg.weights_out}')


## 11 · Post-Training Analysis: Skill Retention vs. Gain

In [ ]:
print('Final comprehensive evaluation...')
cida.eval(); plan_projector.eval()
final_res = eval_split(val_records, n=200, max_new=128)

CODE_TASKS_ALL = {'mbpp','humaneval','codealpaca','evoeval','apps'}
TASK_TYPES = {
    'math':       'Reasoning anchor  (MUST retain)',
    'logic':      'Reasoning anchor  (MUST retain)',
    'mbpp':       'New coding skill',
    'humaneval':  'New coding skill  (gold standard)',
    'codealpaca': 'New coding skill',
    'evoeval':    'New coding skill  (hard variants)',
    'apps':       'New coding skill  (competitive)',
    'chat':       'General (retain)',
}

print()
print('='*65)
print('  CONTINUAL FINE-TUNING v2 — FINAL RESULTS')
print('='*65)
print(f'  {"Task":<16}  {"Accuracy":>10}  Description')
print('-'*65)
for t, acc in sorted(final_res.items()):
    desc = TASK_TYPES.get(t, '')
    print(f'  {t:<16}  {acc*100:>9.1f}%  {desc}')
print('='*65)

# Aggregate stats
math_acc  = final_res.get('math', 0)
logic_acc = final_res.get('logic', 0)
code_vals = [final_res[k] for k in CODE_TASKS_ALL if k in final_res]
code_acc  = sum(code_vals)/len(code_vals) if code_vals else 0
print(f'\n  Reasoning avg   : {(math_acc+logic_acc)/2*100:.1f}%  (math={math_acc*100:.1f}%, logic={logic_acc*100:.1f}%)')
print(f'  Coding avg      : {code_acc*100:.1f}%')
print(f'  Combined score  : {(0.5*(math_acc+logic_acc)/2 + 0.5*code_acc)*100:.1f}%')

# Training curve
if history['total_loss']:
    print(f'\n  Training loss   : {history["total_loss"][0]:.4f} -> {history["total_loss"][-1]:.4f}')
if history['disagreement']:
    print(f'  Disagreement    : {history["disagreement"][0]:.4f} -> {history["disagreement"][-1]:.4f}')

if history['val_step']:
    print('\n  Validation curve:')
    for s, res in zip(history['val_step'], history['val_results']):
        m  = res.get('math', 0)
        cv = [res[k] for k in CODE_TASKS_ALL if k in res]
        c  = sum(cv)/len(cv) if cv else 0
        status = 'OK' if m > 0.35 else 'FORGETTING!'
        print(f'    step {s:>4}: math={m*100:.1f}%  code_avg={c*100:.1f}%  {status}')

print()
print('='*65)
print(f'  Best checkpoint : {cfg.weights_out}/best')
print(f'  Final checkpoint: {cfg.weights_out}/final')
print('='*65)
print()
print('Reload best:')
print(f'  cida_cfg = CIDAPluginConfig.from_pretrained("{cfg.weights_out}/best")')
print(f'  cida     = CIDAPlugin(cida_cfg)')
print(f'  sd       = torch.load("{cfg.weights_out}/best/pytorch_model.bin")')
print(f'  cida.load_state_dict(sd)')
